# Task 3 — Heritage Crack Segmentation

**Model:** U-Net + ResNet50 encoder (ImageNet pretrained)  
**Datasets:** Masonry (240 images) + CrackForest (118 images) = 358 total  
**Target:** mIoU > 80%  
**Loss:** BCE + Dice (combined)  

**Runtime required:** T4 GPU (`Runtime > Change runtime type > T4 GPU`)

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
!pip install -q segmentation-models-pytorch albumentations==1.4.3

## 1. Mount Drive & Prepare Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/HeritagePreservation')
DATA_DIR  = Path('/content/data')
CKPT_DIR  = Path('/content/checkpoints/segmentor')
PLOTS_DIR = Path('/content/plots/segmentor')

for d in [DATA_DIR, CKPT_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive contents:")
for f in sorted(DRIVE_DIR.iterdir()):
    print(f"  {f.name}")

In [ ]:
import zipfile

for zip_name in ['masonry.zip', 'crackforest.zip']:
    zip_path = DRIVE_DIR / zip_name
    out_name = zip_name.replace('.zip', '')
    out_dir  = DATA_DIR / out_name
    if out_dir.exists() and any(out_dir.rglob('*.*')):
        print(f"{zip_name}: already unzipped, skipping.")
    else:
        print(f"Unzipping {zip_name} ({zip_path.stat().st_size/1e6:.1f} MB)...")
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(DATA_DIR)
        print("  Done.")

In [ ]:
MASONRY_DIR    = DATA_DIR / 'masonry'
CRACKFOREST_DIR = DATA_DIR / 'crackforest'

# Verify structure: images/*.png|jpg  masks/*.png
for name, d in [('masonry', MASONRY_DIR), ('crackforest', CRACKFOREST_DIR)]:
    imgs  = list((d / 'images').glob('*.*'))
    masks = list((d / 'masks').glob('*.*'))
    print(f"{name}: {len(imgs)} images, {len(masks)} masks")
    print(f"  sample image : {imgs[0].name}")
    print(f"  sample mask  : {masks[0].name}")

## 2. Dataset & Transforms

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
IMG_SIZE      = 256


def get_seg_transforms(split):
    if split == 'train':
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1, p=0.5),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])


def collect_pairs(dataset_dirs):
    """Collect (image_path, mask_path) pairs from one or more dataset dirs."""
    pairs = []
    for d in dataset_dirs:
        img_dir  = Path(d) / 'images'
        mask_dir = Path(d) / 'masks'
        for img_path in sorted(img_dir.glob('*.*')):
            stem = img_path.stem
            mask_path = mask_dir / f"{stem}.png"
            if mask_path.exists():
                pairs.append((img_path, mask_path))
    return pairs


class CrackSegDataset(Dataset):
    def __init__(self, pairs, split='train', transform=None):
        self.pairs     = pairs
        self.transform = transform or get_seg_transforms(split)

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img  = cv2.imread(str(img_path))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)  # binary 0/1

        out = self.transform(image=img, mask=mask)
        return out['image'], out['mask'].unsqueeze(0)  # (C,H,W), (1,H,W)

print("Dataset class ready.")

## 3. Model — U-Net + ResNet50

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

def build_unet(encoder='resnet50', pretrained=True):
    return smp.Unet(
        encoder_name=encoder,
        encoder_weights='imagenet' if pretrained else None,
        in_channels=3,
        classes=1,
        activation=None,   # raw logits; apply sigmoid in loss/metric
    )

print("U-Net factory ready.")

## 4. Loss & Metrics

In [ ]:
import torch

bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def combined_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)


def compute_seg_metrics(pred_logits, target):
    """Returns dict with iou and dice (micro-averaged over batch)."""
    pred_binary = (torch.sigmoid(pred_logits) > 0.5).long()
    target_long = target.long()
    tp, fp, fn, tn = smp.metrics.get_stats(
        pred_binary, target_long, mode='binary')
    return {
        'iou':  float(smp.metrics.iou_score(tp, fp, fn, tn, reduction='micro')),
        'dice': float(smp.metrics.f1_score( tp, fp, fn, tn, reduction='micro')),
    }

print("Loss and metrics ready.")

## 5. Training

In [ ]:
import random

# ── Config ───────────────────────────────────────────────────────────────────
ENCODER     = 'resnet50'
EPOCHS      = 60
BATCH_SIZE  = 16
LR          = 1e-4
SEED        = 42
NUM_WORKERS = 2
# ─────────────────────────────────────────────────────────────────────────────

DEVICE = torch.device('cuda')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"Encoder : {ENCODER}")
print(f"Input   : {IMG_SIZE}x{IMG_SIZE}")
print(f"Epochs  : {EPOCHS}")
print(f"Batch   : {BATCH_SIZE}")
print(f"Device  : {DEVICE}")

In [ ]:
from torch.utils.data import DataLoader

all_pairs = collect_pairs([MASONRY_DIR, CRACKFOREST_DIR])
print(f"Total pairs: {len(all_pairs)}")

train_p, tmp_p = train_test_split(all_pairs, test_size=0.30, random_state=SEED)
val_p,  test_p = train_test_split(tmp_p,     test_size=0.50, random_state=SEED)
print(f"Train: {len(train_p)}  Val: {len(val_p)}  Test: {len(test_p)}")

train_ds = CrackSegDataset(train_p, split='train')
val_ds   = CrackSegDataset(val_p,   split='val')
test_ds  = CrackSegDataset(test_p,  split='test')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

In [ ]:
model     = build_unet(ENCODER).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")

In [ ]:
from tqdm.notebook import tqdm
from torch.amp import GradScaler, autocast

scaler = GradScaler('cuda')


def run_seg_epoch(model, loader, optimizer, train):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_iou, all_dice = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, masks in tqdm(loader, leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)

            with autocast('cuda'):
                preds = model(imgs)
                loss  = combined_loss(preds, masks)

            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            total_loss += loss.item() * len(imgs)
            m = compute_seg_metrics(preds.float(), masks)
            all_iou.append(m['iou'])
            all_dice.append(m['dice'])

    return {
        'loss': total_loss / len(loader.dataset),
        'iou':  float(np.mean(all_iou)),
        'dice': float(np.mean(all_dice)),
    }


history  = {k: [] for k in ['train_loss', 'train_iou', 'val_loss', 'val_iou', 'val_dice']}
best_iou = 0.0
torch.cuda.empty_cache()

for epoch in range(1, EPOCHS + 1):
    tr = run_seg_epoch(model, train_loader, optimizer, train=True)
    va = run_seg_epoch(model, val_loader,   optimizer, train=False)
    scheduler.step()

    for k, v in [('train_loss', tr['loss']), ('train_iou', tr['iou']),
                 ('val_loss',   va['loss']), ('val_iou',   va['iou']),
                 ('val_dice',   va['dice'])]:
        history[k].append(v)

    print(f"[{epoch:02d}/{EPOCHS}] "
          f"loss={tr['loss']:.4f} iou={tr['iou']:.4f} | "
          f"val_loss={va['loss']:.4f} val_iou={va['iou']:.4f} val_dice={va['dice']:.4f}")

    if va['iou'] > best_iou:
        best_iou = va['iou']
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_iou': best_iou, 'val_dice': va['dice'],
                    'encoder': ENCODER, 'img_size': IMG_SIZE},
                   CKPT_DIR / 'best.pth')
        print(f"   -> best saved (val_iou={best_iou:.4f})")

torch.save({'epoch': EPOCHS, 'model_state': model.state_dict(),
            'encoder': ENCODER}, CKPT_DIR / 'last.pth')
print("\nTraining complete!")

In [ ]:
import matplotlib.pyplot as plt
import json

epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='train')
axes[0].plot(epochs, history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs, history['train_iou'], label='train')
axes[1].plot(epochs, history['val_iou'],   label='val')
axes[1].axhline(0.80, color='r', linestyle='--', label='target 0.80')
axes[1].set_title('IoU'); axes[1].legend()

axes[2].plot(epochs, history['val_dice'], label='Dice', color='green')
axes[2].set_title('Val Dice'); axes[2].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'training_curves.png', dpi=150)
plt.show()

## 6. Test Evaluation

In [ ]:
ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_m = run_seg_epoch(model, test_loader, optimizer=None, train=False)

print("=== Test Results ===")
print(f"mIoU  : {test_m['iou']:.4f}  (target >0.80)")
print(f"Dice  : {test_m['dice']:.4f}")
print(f"Loss  : {test_m['loss']:.4f}")

history['test'] = test_m
with open(CKPT_DIR / 'history.json', 'w') as f:
    json.dump(history, f, indent=2)

In [ ]:
# Visualize 4 predictions: image | ground truth | prediction
import random

sample_indices = random.sample(range(len(test_ds)), 4)
fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Segmentation Results  |  Image | GT Mask | Predicted Mask')

inv_mean = np.array(IMAGENET_MEAN)
inv_std  = np.array(IMAGENET_STD)

model.eval()
for row, idx in enumerate(sample_indices):
    img_t, mask_t = test_ds[idx]

    with torch.no_grad():
        pred = torch.sigmoid(model(img_t.unsqueeze(0).to(DEVICE)))
    pred_bin = (pred.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

    # Denormalize image
    img_np = img_t.permute(1, 2, 0).numpy()
    img_np = (img_np * inv_std + inv_mean).clip(0, 1)

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('Image', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(mask_t.squeeze().numpy(), cmap='gray')
    axes[row, 1].set_title('Ground Truth', fontsize=9)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(pred_bin, cmap='gray')
    axes[row, 2].set_title('Prediction', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(PLOTS_DIR / 'predictions.png', dpi=150)
plt.show()

## 7. Save to Drive

In [ ]:
import shutil

DRIVE_CKPT  = DRIVE_DIR / 'checkpoints' / 'segmentor'
DRIVE_PLOTS = DRIVE_DIR / 'plots' / 'segmentor'
DRIVE_CKPT.mkdir(parents=True,  exist_ok=True)
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

for f in CKPT_DIR.iterdir():
    shutil.copy2(f, DRIVE_CKPT / f.name)

for f in PLOTS_DIR.iterdir():
    shutil.copy2(f, DRIVE_PLOTS / f.name)

print("Saved checkpoints:")
for f in sorted(DRIVE_CKPT.iterdir()):
    print(f"  {f.name:30s}  {f.stat().st_size/1e6:.1f} MB")

print(f"\nFinal test metrics:")
print(f"  mIoU : {test_m['iou']:.4f}  (target >0.80)")
print(f"  Dice : {test_m['dice']:.4f}")